In [1]:
import math
import torch
from torch import nn
from torch.nn import functional as F
from torchvision import datasets, transforms, utils
from torch.utils.data import DataLoader
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 엡실론 모델
디퓨전 모델은 노이즈를 지우는 모델이 아닌, 실제로는 특정 단계에서 얼마나 많은 가우시안 노이즈가 추가되었는지 예측하는 모델입니다. 주로 U-net을 사용함.

이 노이즈를 예측하여, 점진적으로 가우시안 노이즈를 순차적으로 제거해 이미지를 재구성하는 것이 목적입니다.

해당 모델을 학습할 때, 타임스텝 t에 대해 처음 이미지에서 여기까지 더해진 모든 노이즈를 예측합니다. 이 타임스텝에 대한 정보는 인코딩으로 들어가 있습니다.

In [ ]:
# 매우 매우 작은 Noise 예측 UNet
class TinyUNet(nn.Module):
    def __init__(self, img_channels: int, base_channels: int = 64):
        super().__init__()
        self.down1 = nn.Conv2d(img_channels, base_channels, 3, padding=1)
        self.down2 = nn.Conv2d(base_channels, base_channels * 2, 3, stride=2, padding=1)
        self.mid  = nn.Conv2d(base_channels * 2, base_channels * 2, 3, padding=1)
        self.up1  = nn.ConvTranspose2d(base_channels * 2, base_channels, 4, stride=2, padding=1)
        self.out  = nn.Conv2d(base_channels, img_channels, 3, padding=1)

        # timestep t를 임베딩. 간단한 sinusoid로도 대체 가능.
        self.time_mlp = nn.Sequential(
            nn.Linear(1, base_channels * 2), nn.SiLU(), nn.Linear(base_channels * 2, base_channels * 2)
        )

    def forward(self, x, t):
        # t ∈ [0, T‑1], t는 피쳐 벡터로 만들어두고 각 spatial dims로 브로드캐스팅 하기.
        t = self.time_mlp(t[:, None].float())               # (B, C)
        t = t[..., None, None]                              # (B, C, 1, 1)
        y = F.silu(self.down1(x) + t[:, :64])
        y = F.silu(self.down2(y) + t)
        y = F.silu(self.mid(y)  + t)
        y = F.silu(self.up1(y)   + t[:, :64])
        return self.out(y)                                  # 노이즈 ε 최종적으로 예측.

# 하이퍼파라미터
## 스텝
논문에서는 1000 스텝을 활용해 기존 이미지를 거의 완벽한 가우시안 노이즈로 변경합니다.
## 스케줄링 설정
타임스텝에 따라 선형적으로 증가하고, 감소하는 스케줄링 파라미터 $\beta$와 1에서 $\beta$를 뺀 $\alpha$를 구해둡니다.

또한, 미리 한번에 1000번 적용된 노이즈를 사용하기 위해 한번에 모든 타임스텝의 $\alpha$를 한번에 곱해둔 cumprod $\bar\alpha$ 또한 구해둡니다.

그리고 또 식에서 계속 재사용될 것들은 그냥 미리 루트 씌워두고요.

In [ ]:
# 2. 하이퍼파라미터 설정----------------------------------------
T = 1000                     # 1000번 노이즈 먹입니다.

# Linear 스케줄링 베타 값 βₜ ∈ (1e‑4, 0.02)(4번 식)
beta_start, beta_end = 1e-4, 0.02
beta = torch.linspace(beta_start, beta_end, T, device=DEVICE)   # (T,)
alpha = 1.0 - beta                                               # αₜ (2번 식)
alpha_bar = torch.cumprod(alpha, dim=0)                          # αₜ Bar (한번에 다 곱해서 한번의 forward noising process로 처리하기 위해 함)

# 어짜피 알파, 알파 바 둘다 루트 씌워서 계산한거를 계속 사용하므로, 미리 만들어 버리기
sqrt_alpha_bar      = torch.sqrt(alpha_bar)
sqrt_one_minus_ab   = torch.sqrt(1.0 - alpha_bar)

# 주어진 텐서에 대해 t번째 값을 가져오는 단순 함수. timestep 관련된거 편하게 하는 것이 전부.
get = lambda v, t, x: v[t].reshape(x.shape[0], 1, 1, 1)

# $q(x_t|x_0)$
처음 이미지를 가지고 timestep t 만큼 가우시안 노이즈가 적용된 이미지를 생성합니다.

앞서 구한 cumprod 된 $\bar\alpha$를 활용해 한번에 구하는 것이 특징이구요. 아무 시점 t에 대해 사용할 수 있습니다.


In [ ]:
# 3. 한번에 timestep t로 넘어가는 노이즈 생성 함수 q ----------------------------------------
# 재파라미터화 전략 사용 (2.2)
#     xₜ = √\bar{αₜ} · x₀  +  √(1‑\bar{αₜ}) · ε, ε~N(0,I)
# 엡실론이 재파라미터화 전략의 그것이다.

def q_sample(x0, t, noise=None):
    """Diffuse the data (forward process) to an arbitrary timestep t."""
    if noise is None:
        noise = torch.randn_like(x0)
    return get(sqrt_alpha_bar, t, x0) * x0 + get(sqrt_one_minus_ab, t, x0) * noise

# 손실 함수
엡실론 모델, 즉 노이즈 예측 모델 U-net을 학습시킵니다.

q_sample 함수를 이용해 랜덤하게 샘플한 $\epsilon$(가우시안 노이즈)를 적용시키고, 이렇게 만든 노이지한 이미지를 그대로 U-Net 모델에 타임스텝 값과 함께 제공합니다.

이제 U-Net 모델은 제공된 거의 순수 노이즈인 이미지를 가지고, 원본에 추가된 **노이즈만** 예측합니다.
여기서 엡실론 모델이라는 이름이 명확해 집니다. 추가된 노이즈 $\epsilon$을 근사해야 하기 때문이죠.

이것은 매우 중요합니다! 원본 이미지를 그대로 예측하는 것은 사실상 불가능합니다. 또한 원본 이미지를 제공하고 노이즈를 예측하는 방식은 생성형 모델에는 적합하지 않는 방식이죠. 그냥 VAE랑 다를 것이 없습니다.

# ELBO 유도

## 1단계: 로그 우도 시작점

DDPM은 잠재 변수 모델로, 관측된 데이터 $x_0$의 로그 우도는 다음과 같이 표현됩니다:

$$
\log p_\theta(x_0) = \log \int p_\theta(x_{0:T}) \, dx_{1:T}
$$

여기서:

* $p_\theta(x_{0:T})$는 역방향 과정의 결합 확률 분포입니다.
* $x_{1:T}$는 잠재 변수로, 직접 관측되지 않지만 모델링에 포함됩니다.([Medium][1])

이 적분은 직접 계산하기 어렵기 때문에, 변분 추론을 사용하여 하한을 설정합니다.

---

## 2단계: 변분 추론을 통한 ELBO 도입

변분 추론에서는 실제 분포 $p_\theta(x_{1:T} | x_0)$를 근사하기 위해 $q(x_{1:T} | x_0)$를 도입합니다. 이를 통해 로그 우도를 다음과 같이 표현할 수 있습니다:

$$
\log p_\theta(x_0) = \log \int q(x_{1:T} | x_0) \cdot \frac{p_\theta(x_{0:T})}{q(x_{1:T} | x_0)} \, dx_{1:T}
$$

여기서 분자와 분모에 동일한 $q(x_{1:T} | x_0)$를 곱하고 나눈 이유는, 적분 내에 몬테카를로 샘플링이 가능한 분포를 도입하여 계산을 용이하게 하기 위함입니다.

---

## 3단계: Jensen 부등식 적용

Jensen의 부등식을 적용하면 다음과 같은 변분 하한(ELBO)을 얻을 수 있습니다:

$$
\log p_\theta(x_0) \geq \mathbb{E}_{q(x_{1:T} | x_0)} \left[ \log \frac{p_\theta(x_{0:T})}{q(x_{1:T} | x_0)} \right] =: \mathcal{L}_{\text{ELBO}}
$$

이 식에서 기대값 $\mathbb{E}_{q(x_{1:T} | x_0)}$는 $q(x_{1:T} | x_0)$에 따라 샘플링된 $x_{1:T}$에 대한 평균을 의미합니다.

---

## 4단계: 결합 확률 분포의 분해

결합 확률 분포 $p_\theta(x_{0:T})$와 $q(x_{1:T} | x_0)$를 다음과 같이 분해합니다:([NeurIPS Proceedings][2])

* $p_\theta(x_{0:T}) = p(x_T) \prod_{t=1}^T p_\theta(x_{t-1} | x_t)$
* $q(x_{1:T} | x_0) = \prod_{t=1}^T q(x_t | x_{t-1})$

이를 통해 ELBO는 다음과 같이 표현됩니다:

$$
\mathcal{L}_{\text{ELBO}} = \mathbb{E}_{q(x_{1:T} | x_0)} \left[ \log p(x_T) + \sum_{t=1}^T \log p_\theta(x_{t-1} | x_t) - \sum_{t=1}^T \log q(x_t | x_{t-1}) \right]
$$

---

## 5단계: KL 다이버전스로 표현

ELBO를 KL 다이버전스의 합으로 표현하면 다음과 같습니다:([Georgia Tech Sites][3])

$$
\mathcal{L}_{\text{ELBO}} = -D_{\text{KL}}(q(x_T | x_0) \| p(x_T)) - \sum_{t=2}^T D_{\text{KL}}(q(x_{t-1} | x_t, x_0) \| p_\theta(x_{t-1} | x_t)) + \mathbb{E}_{q(x_1 | x_0)}[\log p_\theta(x_0 | x_1)]
$$

여기서 각 KL 다이버전스 항은 다음을 의미합니다:

* $D_{\text{KL}}(q(x_T | x_0) \| p(x_T))$: 최종 단계에서의 노이즈 분포와 모델의 초기 분포 간의 차이.
* $D_{\text{KL}}(q(x_{t-1} | x_t, x_0) \| p_\theta(x_{t-1} | x_t))$: 각 단계에서의 실제 역방향 분포와 모델이 학습한 분포 간의 차이.
* $\mathbb{E}_{q(x_1 | x_0)}[\log p_\theta(x_0 | x_1)]$: 첫 번째 단계에서의 복원 정확도.

---

## 6단계: 최종 손실 함수

모델 학습을 위한 최종 손실 함수는 다음과 같이 정의됩니다:

$$
\mathcal{L}_{\text{loss}} = D_{\text{KL}}(q(x_T | x_0) \| p(x_T)) + \sum_{t=2}^T D_{\text{KL}}(q(x_{t-1} | x_t, x_0) \| p_\theta(x_{t-1} | x_t)) - \mathbb{E}_{q(x_1 | x_0)}[\log p_\theta(x_0 | x_1)]
$$

이 손실 함수는 모델이 각 단계에서의 노이즈 제거를 효과적으로 학습하도록 유도합니다.

---

## 추가 설명: 분수 항의 도입 이유

ELBO 유도 과정에서 $\frac{q(x_{1:T} | x_0)}{q(x_{1:T} | x_0)}$를 곱하는 이유는, 계산이 어려운 $p_\theta(x_{0:T})$를 샘플링 가능한 분포 $q(x_{1:T} | x_0)$를 통해 근사하기 위함입니다. 이를 통해 기대값을 계산할 수 있으며, 모델 학습이 가능해집니다.

---
더 자세한 수식 유도와 설명은 다음 자료에서 확인하실 수 있습니다:

* [Denoising Diffusion Probabilistic Models - arXiv](https://arxiv.org/pdf/2006.11239)
* [Diffusion Models + Variational Inference - CMU Slides](https://www.cs.cmu.edu/~mgormley/courses/10423-s24//slides/lecture8-diffusion-ink.pdf)([arXiv][4], [CMU School of Computer Science][5])

이러한 자료들은 LaTeX 형식의 수식과 함께 ELBO의 유도 과정을 상세히 설명하고 있어, DDPM의 수학적 기반을 이해하는 데 도움이 될 것입니다.

[1]: https://medium.com/%40ml.swlee/paper-review-denoising-diffusion-probabilistic-models-29dbcee90db8?utm_source=chatgpt.com "Paper review: Denoising Diffusion Probabilistic Models - Medium"
[2]: https://proceedings.neurips.cc/paper/2021/file/cfe8504bda37b575c70ee1a8276f3486-Supplemental.pdf?utm_source=chatgpt.com "[PDF] A Details of denoising diffusion probabilistic models B Algorithms"
[3]: https://sites.cc.gatech.edu/classes/AY2023/cs7643_spring/slides/L26_GenerativeModels2.pdf?utm_source=chatgpt.com "[PDF] Denoising Diffusion Probabilistic Models (DDPMs) Slides adapted ..."
[4]: https://arxiv.org/pdf/2006.11239?utm_source=chatgpt.com "[PDF] Denoising Diffusion Probabilistic Models - arXiv"
[5]: https://www.cs.cmu.edu/~mgormley/courses/10423-s24//slides/lecture8-diffusion-ink.pdf?utm_source=chatgpt.com "[PDF] Diffusion Models + Variational Inference"



In [ ]:
# 4. Training loss -----------------------------------------------------

def p_losses(model, x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    xt = q_sample(x0, t, noise)
    eps_hat = model(xt, t.float())
    return F.mse_loss(eps_hat, noise)

In [ ]:
# 5. 노이즈 제거 과정이랑 루프 p_θ(xₜ₋₁|xₜ) -----------------------------------
#     μ_θ(xₜ,t) = 1/√αₜ · \bigl(xₜ - βₜ / √(1-\bar{αₜ}) · ε_θ(xₜ,t)\bigr)

@torch.no_grad()
def p_sample(model, xt, t):
    beta_t      = get(beta, t, xt)
    sqrt_ab_t   = get(sqrt_alpha_bar, t, xt)
    sqrt_oab_t  = get(sqrt_one_minus_ab, t, xt)

    eps_hat = model(xt, t.float())
    mu      = (1.0 / torch.sqrt(get(alpha, t, xt))) * (xt - beta_t / sqrt_oab_t * eps_hat)

    if t == 0:
        return mu  # 최종적인 이미지 reconstruct에서는 variance 없이, 명확한 이미지를 생성.
    else:
        noise = torch.randn_like(xt)
        sigma = torch.sqrt(beta_t)
        return mu + sigma * noise

@torch.no_grad()
def p_sample_loop(model, shape):
    xt = torch.randn(shape, device=DEVICE)  # x_T ~ N(0,I)
    imgs = []
    for t in reversed(range(T)):
        xt = p_sample(model, xt, torch.full((shape[0],), t, device=DEVICE, dtype=torch.long))
        if t % 100 == 0 or t == T-1:
            imgs.append(xt.cpu())
    return imgs

In [ ]:
# 6. 훈련 루프-------------------------------------------
transform = transforms.Compose([
    transforms.Resize(32),
    transforms.CenterCrop(32),
    transforms.ToTensor(),
    lambda x: x * 2.0 - 1.0, # 정규화
])
train_data = datasets.MNIST(root="./data", download=True, transform=transform)
loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=4, drop_last=True)

model = TinyUNet(img_channels=1).to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=2e-4)

epochs = 1
for epoch in range(epochs):
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for x0, _ in pbar:
        x0 = x0.to(DEVICE)
        t  = torch.randint(0, T, (x0.size(0),), device=DEVICE, dtype=torch.long)
        loss = p_losses(model, x0, t)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        pbar.set_postfix(loss=loss.item())

    # Sample & save a grid after each epoch
    sampled = p_sample_loop(model, (64, 1, 32, 32))[-1]
    grid = utils.make_grid((sampled + 1) / 2)  # back to [0,1]
    utils.save_image(grid, f"samples_epoch{epoch+1}.png")

print("끝")
